<a href="https://colab.research.google.com/github/msquareddd/hugging-face-smol-course/blob/notebooks/hf_jobs_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing dependencies

In [ ]:
!pip install -U -q "huggingface_hub[cli]"

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# Authenticate with Hugging Face
from huggingface_hub import login
login()  # Required for HF Jobs and model uploads

In [ ]:
import time, datetime
from datetime import date
now = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

# Unit 1 - SFT

In [ ]:
# Create the evaluation script
script_content = """
import subprocess
import sys

# Run lighteval with vllm
subprocess.run([
    "lighteval", "vllm",
    "model_name=msquaredd/SmolLM3-Custom-SFT-20250910143319",
    "lighteval|gsm8k|0|0",
    "--push-to-hub",
    "--results-org", "msquaredd"
], check=True)
"""

# Save it to a file
with open('run_eval.py', 'w') as f:
    f.write(script_content)

In [ ]:
# Run the command
!hf jobs uv run \
  --flavor a10g-large \
  --with "lighteval[vllm]" \
  -s HF_TOKEN \
  run_eval.py

# Unit 2 - DPO

## DPO Training with hf jobs

In [ ]:
# Use TRL's DPO script with HF Jobs
!hf jobs uv run \
    --flavor a100-large \
    --timeout 3h \
    --secrets HF_TOKEN \
    "https://raw.githubusercontent.com/huggingface/trl/main/trl/scripts/dpo.py" \
    --model_name_or_path HuggingFaceTB/SmolLM3-3B \
    --dataset_name Anthropic/hh-rlhf \
    --learning_rate 5e-7 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_steps 1000 \
    --beta 0.1 \
    --max_prompt_length 512 \
    --max_length 1024 \
    --output_dir smollm3-dpo-aligned \
    --push_to_hub \
    --hub_model_id msquaredd/smollm3-dpo-aligned-202509291110\
    --report_to trackio

In [ ]:
# Check job details
!hf jobs inspect 68da4e5eba2a26256a3d768a

In [ ]:
# msquaredd/smollm3-dpo-aligned-202509291110

## Eval

In [ ]:
!pip install -q "lighteval[vllm,multilingual]"

In [ ]:
!lighteval tasks list

In [ ]:
# lighteval|truthfulqa:gen lighteval|gsm8k

In [ ]:
# Old

!hf jobs uv run \
    --flavor a10g-large \
    --with "lighteval[vllm]" \
    --secrets HF_TOKEN \
    lighteval vllm "model_name=msquaredd/smollm3-dpo-aligned-202509291110" \
    "lighteval|truthfulqa:mc2|0|0,lighteval|hellaswag|0|0,lighteval|arc:challenge|0|0" \
    --push-to-hub --results-org msquaredd

In [ ]:
# you should change your task name to "suite|task|num_fewshot"

In [ ]:
# New

!hf jobs uv run \
    --flavor a10g-large \
    --timeout 3h \
    --with "lighteval[vllm], emoji" \
    --secrets HF_TOKEN \
    lighteval vllm "model_name=msquaredd/smollm3-dpo-aligned-202509291110" \
    "leaderboard|truthfulqa:mc|0|0,leaderboard|hellaswag|0|0,leaderboard|arc:challenge|0|0" \
    --push-to-hub --results-org msquaredd

In [ ]:
# 68da8eb4ba2a26256a3d7727
# 68db7c2965795a61a80d7764

In [ ]:
!hf jobs ps --all

In [ ]:
!hf jobs inspect 68db7c2965795a61a80d7764

# Unit 3 - VLMs

In [ ]:
now

## SFT SmolVLM

In [ ]:
# Use TRL's maintained SFT script directly
!hf jobs uv run \
    --flavor a10g-large \
    --timeout 3h \
    --secrets HF_TOKEN \
    --with num2words==0.5.14 \
    "https://raw.githubusercontent.com/huggingface/trl/main/trl/scripts/sft.py" \
    --model_name_or_path HuggingFaceTB/SmolVLM2-2.2B-Instruct \
    --dataset_name trl-lib/llava-instruct-mix\
    --learning_rate 5e-5 \
    --per_device_train_batch_size 4 \
    --max_length -1 \
    --max_steps 1000 \
    --output_dir smolvlm2-2.2b-instruct-sft-jobs \
    --push_to_hub \
    --hub_model_id msquaredd/smolvlm2-2.2b-instruct-sft-jobs-20250930122121 \
    --report_to trackio

In [ ]:
# Check job details
!hf jobs inspect 68dbcb5b65795a61a80d77ca